In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, window, sum as spark_sum, count
import os
import shutil
import time
import pandas as pd

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# 2. Menyiapkan folder simulasi streaming
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# 3. Mendefinisikan skema eksplisit untuk data pesanan Kopi Nusantara
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# 4. Mendefinisikan readStream
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming Source untuk Bagian A siap.")


Streaming Source untuk Bagian A siap.


In [14]:
from pyspark.sql.functions import col, sum as spark_sum, count

# 1. Menambahkan kolom turunan total_harga (qty * harga_satuan)
df_stream_with_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# 2. Melakukan agregasi per nama_cabang
agregasi_cabang = df_stream_with_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# --- PENAMBAHAN: Hentikan query "agregasi_cabang" jika sedang aktif ---
for stream in spark.streams.active:
    if stream.name == "agregasi_cabang":
        stream.stop()

# 3. Menjalankan streaming query (sink: memory)
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming Query Agregasi per Cabang (Bagian B) berhasil berjalan.")

Streaming Query Agregasi per Cabang (Bagian B) berhasil berjalan.


In [15]:
# 1. Membaca ulang stream untuk query kedua
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

# 2. Agregasi berbasis window waktu 2 menit pada kolom waktu_pesan
agregasi_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_omset_window"),
    count("pesanan_id").alias("jumlah_pesanan_window")
)

# 3. Menjalankan streaming query (sink: memory)
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming Query Window Waktu 2 Menit (Bagian C) berhasil berjalan.")

Streaming Query Window Waktu 2 Menit (Bagian C) berhasil berjalan.


In [16]:
from datetime import datetime

# --- BATCH 1 ---
pd.DataFrame({
    "pesanan_id": ["P101", "P102", "P103"],
    "nama_cabang": ["Cabang Malioboro", "Cabang Simpang Lima", "Cabang Malioboro"],
    "nama_menu": ["Kopi Kenangan", "Matcha Latte", "Americano"],
    "qty": [2, 1, 3],
    "harga_satuan": [18000, 22000, 15000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 terkirim. Memproses...")
time.sleep(6)
print("=== HASIL AGREGASI CABANG (SETELAH BATCH 1) ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()


# --- BATCH 2 ---
pd.DataFrame({
    "pesanan_id": ["P104", "P105"],
    "nama_cabang": ["Cabang Simpang Lima", "Cabang Malioboro"],
    "nama_menu": ["Roti Bakar", "Kopi Milk"],
    "qty": [2, 2],
    "harga_satuan": [12000, 20000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 terkirim. Memproses...")
time.sleep(6)
print("=== HASIL AGREGASI CABANG (SETELAH BATCH 2) ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()


# --- BATCH 3 ---
pd.DataFrame({
    "pesanan_id": ["P106", "P107", "P108"],
    "nama_cabang": ["Cabang Malioboro", "Cabang Simpang Lima", "Cabang Tidar"],
    "nama_menu": ["Espresso", "Kopi Kenangan", "V60"],
    "qty": [1, 3, 2],
    "harga_satuan": [15000, 18000, 25000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 terkirim. Memproses...")
time.sleep(6)
print("=== HASIL AGREGASI CABANG (SETELAH BATCH 3) ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

print("=== HASIL AGREGASI WINDOW WAKTU 2 MENIT ===")
spark.sql("SELECT window.start, window.end, total_omset_window, jumlah_pesanan_window FROM agregasi_waktu").show(truncate=False)

Batch 1 terkirim. Memproses...


=== HASIL AGREGASI CABANG (SETELAH BATCH 1) ===
+-------------------+----------------+--------------+
|        nama_cabang|total_pendapatan|jumlah_pesanan|
+-------------------+----------------+--------------+
|   Cabang Malioboro|           81000|             2|
|Cabang Simpang Lima|           22000|             1|
+-------------------+----------------+--------------+

Batch 2 terkirim. Memproses...
=== HASIL AGREGASI CABANG (SETELAH BATCH 2) ===
+-------------------+----------------+--------------+
|        nama_cabang|total_pendapatan|jumlah_pesanan|
+-------------------+----------------+--------------+
|   Cabang Malioboro|          121000|             3|
|Cabang Simpang Lima|           46000|             2|
+-------------------+----------------+--------------+

Batch 3 terkirim. Memproses...
=== HASIL AGREGASI CABANG (SETELAH BATCH 3) ===
+-------------------+----------------+--------------+
|        nama_cabang|total_pendapatan|jumlah_pesanan|
+-------------------+---------------

In [18]:
# 1. Bersihkan folder output & checkpoint sebelum memulainya
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# 2. Menjalankan streaming query ketiga (Output mentah ke Parquet)
query_file = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming Query ke Sink Parquet (Bagian E) dimulai.")

# 3. Mengirim 1 batch data tambahan
pd.DataFrame({
    "pesanan_id": ["P109"],
    "nama_cabang": ["Cabang Tidar"],
    "nama_menu": ["Croissant"],
    "qty": [1],
    "harga_satuan": [20000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")]
}).to_csv(f"{folder_tugas}/batch_4_parquet.csv", index=False)

print("Batch 4 terkirim untuk menguji Parquet Sink. Menunggu...")
time.sleep(7)

# 4. Verifikasi file Parquet tersimpan di disk
print("=== VERIFIKASI ISI FOLDER OUTPUT PARQUET ===")
!ls tugas10_output

# 5. Membaca kembali file Parquet sebagai DataFrame batch biasa
df_parquet_read = spark.read.parquet("tugas10_output")
print("=== HASIL BACA DATAFRAME PARQUET ===")
df_parquet_read.show()

Streaming Query ke Sink Parquet (Bagian E) dimulai.
Batch 4 terkirim untuk menguji Parquet Sink. Menunggu...
=== VERIFIKASI ISI FOLDER OUTPUT PARQUET ===
part-00000-270a16cb-1f29-4279-91c4-fa88c583c128-c000.snappy.parquet
part-00001-f349b9db-0303-4a96-a832-00eaaedb6461-c000.snappy.parquet
part-00002-c3fd4f2e-f249-450f-bc94-63f4eeb9f073-c000.snappy.parquet
part-00003-06c4eeda-3c72-4c53-961b-7d378de1c888-c000.snappy.parquet
_spark_metadata
=== HASIL BACA DATAFRAME PARQUET ===
+----------+-------------------+-------------+---+------------+-------------------+
|pesanan_id|        nama_cabang|    nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-------------------+-------------+---+------------+-------------------+
|      P101|   Cabang Malioboro|Kopi Kenangan|  2|       18000|2026-11-05 16:30:49|
|      P102|Cabang Simpang Lima| Matcha Latte|  1|       22000|2026-11-05 16:30:49|
|      P103|   Cabang Malioboro|    Americano|  3|       15000|2026-11-05 16:30:49|
|      P104|Caban